In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, time
import numpy as np
import pandas as pd
from astropy.timeseries import BoxLeastSquares
from scipy.optimize import minimize
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
CASE_ROWS = np.array([4923, 5192, 27897, 27901, 40099, 40114, 63014, 63018, 86891, 87483, 89898, 89906], dtype=int)

sources = []
for mp in INPUT.rglob("simulation_metadata.json"):
    if mp.parent.name != "binary":
        continue
    try:
        meta = json.loads(mp.read_text(encoding="utf-8"))
    except Exception as exc:
        print("Skipping unreadable metadata:", mp, exc)
        continue
    if meta.get("family") == "binary" and int(meta.get("parameter_draws_unique", 0)) >= 100_000:
        sources.append((meta.get("created_utc", ""), mp.parent, meta))
if not sources:
    raise FileNotFoundError("Attach the Day 7 Cosmic swan Bayesian output containing binary observations.npy, parameters.csv, and time_days.npy.")
_, BINARY_DIR, META = sorted(sources, key=lambda x: x[0])[-1]
for fn in ("observations.npy", "parameters.csv", "time_days.npy"):
    if not (BINARY_DIR / fn).is_file():
        raise FileNotFoundError(f"Missing: {BINARY_DIR / fn}")
observations = np.load(BINARY_DIR / "observations.npy", mmap_mode="r")
parameters = pd.read_csv(BINARY_DIR / "parameters.csv")
time_days = np.asarray(np.load(BINARY_DIR / "time_days.npy"), dtype=float)
required = {"period_days", "phase", "depth_primary", "depth_ratio", "duration_fraction", "ellipsoidal_amplitude", "reflection_amplitude", "noise_sigma"}
if not required.issubset(parameters.columns):
    raise ValueError(f"Missing parameter columns: {sorted(required - set(parameters.columns))}")
if observations.shape != (len(parameters), len(time_days)) or CASE_ROWS.max() >= len(parameters):
    raise ValueError("Observation, parameter, time, or fixed case-row dimensions do not match.")

OUT = WORK / ("day9-3-binary-alias-check-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUT.mkdir(parents=True, exist_ok=False)
BASELINE = float(np.ptp(time_days))
MAX_PERIOD = min(13.0, BASELINE / 2.0)
print("[PASS] Input:", BINARY_DIR, "| array:", observations.shape, "| output:", OUT)
print("[PASS] CPU-only; fixed cases:", CASE_ROWS.tolist(), "| no sampling in this cell.")


In [ ]:
BANDS = [(0.1,0.2),(0.2,0.4),(0.4,0.8),(0.8,1.6),(1.6,3.2),(3.2,6.4),(6.4,13.0)]

def clip01(x, sharp=500.0):
    return (np.logaddexp(0.0, sharp*x) - np.logaddexp(0.0, sharp*(x-1.0))) / sharp

def predict(t, p, phase, depth, ratio, duration, ellip, reflection):
    orb = np.mod(t/p - phase + 0.5, 1.0) - 0.5
    sec = np.mod(orb, 1.0) - 0.5
    half = max(duration/2.0, 1e-6)
    ingress = max(half*0.18, 1e-6)
    primary = clip01((half+ingress-np.abs(orb))/(2*ingress))
    secondary = clip01((half+ingress-np.abs(sec))/(2*ingress))
    return 1-depth*primary-depth*ratio*secondary + ellip*np.cos(4*np.pi*orb) + reflection*np.cos(2*np.pi*orb)

def get_modes(t, y):
    bls, modes = BoxLeastSquares(t,y), []
    for lo, hi0 in BANDS:
        hi = min(hi0, MAX_PERIOD)
        if hi <= lo: continue
        dmax = min(1.5, 0.95*lo, 0.12*hi)
        result = bls.autopower(duration=np.geomspace(min(0.005,dmax/2),dmax,16),
                               objective="snr", minimum_period=lo, maximum_period=hi,
                               frequency_factor=12.0)
        power = np.asarray(result.power,float)
        valid = np.flatnonzero(np.isfinite(power))
        if not len(valid): continue
        peaks,_ = find_peaks(power)
        ids = sorted(set(list(peaks)+[int(valid[np.argmax(power[valid])])]),
                     key=lambda i: power[i], reverse=True)
        selected=[]
        for i in ids:
            p=float(result.period[i])
            if any(abs(p-old)/old < .02 for old in selected): continue
            selected.append(p)
            modes.append({"bls_period":p,"power":float(power[i]),
                          "transit_time":float(result.transit_time[i]),
                          "duration_days":float(result.duration[i])})
            if len(selected)==2: break
    if not modes: raise RuntimeError("BLS returned no finite modes.")
    return modes

def aliases(modes):
    out=[]
    for m in modes:
        for factor in (0.5,1.0,2.0):
            p=m["bls_period"]*factor
            if not 0.1 <= p <= MAX_PERIOD: continue
            row={**m,"candidate_period":float(p),"alias_factor":factor}
            old=next((x for x in out if abs(p-x["candidate_period"])/min(p,x["candidate_period"])<0.01),None)
            if old is None: out.append(row)
            elif row["power"]>old["power"]: old.update(row)
    return sorted(out,key=lambda x:x["power"],reverse=True)

def fit_mode(t,y,c):
    p0=c["candidate_period"]
    hw=max(0.04,0.003*p0)
    bounds=[(max(.1,p0-hw),min(MAX_PERIOD,p0+hw)),(0,1),(.002,.45),(0,1),
            (.005,.12),(0,.02),(-.01,.01),(np.log(1e-4),np.log(.03))]
    phase0=float(np.mod(c["transit_time"]/p0,1))
    duration=float(np.clip(c["duration_days"]/p0,.005,.12))
    depth=float(np.clip((np.nanpercentile(y,95)-np.nanpercentile(y,5))/2,.002,.45))
    noise=float(np.clip(np.std(np.diff(y))/np.sqrt(2),1e-4,.03))
    def nll(x):
        pred=predict(t,*x[:7]); ln=x[7]
        return float(.5*np.sum(((y-pred)/np.exp(ln))**2+2*ln+np.log(2*np.pi)))
    starts=[]
    for shift in (0.0,.5):
        x=np.array([p0,(phase0+shift)%1,depth,.5,duration,.005,0.0,np.log(noise)])
        starts.append(np.array([np.clip(v,a+1e-9,b-1e-9) for v,(a,b) in zip(x,bounds)]))
    fits=[minimize(nll,x,method="L-BFGS-B",bounds=bounds,options={"maxiter":1000,"ftol":1e-9}) for x in starts]
    fit=min(fits,key=lambda z: z.fun if np.isfinite(z.fun) else np.inf)
    x=np.asarray(fit.x,float); pred=predict(t,*x[:7]); sig=float(np.exp(x[7]))
    hits=[name for name,v,(a,b) in zip(("period","phase","depth","ratio","duration","ellip","reflection","noise"),x,bounds)
          if min(v-a,b-v)/(b-a)<.005]
    return {"fit_success":bool(fit.success),"fit_period_days":float(x[0]),"phase":float(x[1]),
            "nll":float(fit.fun),"bic":float(2*fit.fun+len(x)*np.log(len(y))),
            "noise_sigma":sig,"rms_over_noise":float(np.sqrt(np.mean((y-pred)**2))/sig),
            "bound_hits":"|".join(hits)}

candidate_rows, case_rows = [], []
for n, row in enumerate(CASE_ROWS,1):
    started=time.time()
    t=np.asarray(time_days,float); y=np.asarray(observations[int(row)],float)
    ok=np.isfinite(t)&np.isfinite(y); t,y=t[ok],y[ok]
    order=np.argsort(t); t,y=t[order],y[order]
    y=y/np.median(y); t=t-t.min()
    modes=get_modes(t,y); cand=aliases(modes); fitted=[]
    print(f"Case {n}/{len(CASE_ROWS)} row {row}: {len(cand)} candidate modes",flush=True)
    truth=float(parameters.iloc[int(row)]["period_days"])
    for c in cand:
        f=fit_mode(t,y,c)
        tol=max(.04,.003*truth)
        fitted.append({**c,**f,"simulation_row":int(row),
                       "known_period_days_for_evaluation":truth,
                       "candidate_mode_matches_truth":bool(abs(f["fit_period_days"]-truth)<=tol)})
    fitted.sort(key=lambda z:z["bic"])
    best_bic=fitted[0]["bic"]
    for rank,r in enumerate(fitted,1):
        r["bic_rank"]=rank; r["delta_bic"]=float(r["bic"]-best_bic)
        candidate_rows.append(r)
    close=[r for r in fitted if r["delta_bic"]<=6]
    best=fitted[0]
    case_rows.append({"simulation_row":int(row),"status":"completed",
        "bls_best_period_days":max(modes,key=lambda z:z["power"])["bls_period"],
        "candidate_count":len(fitted),"best_fit_period_days":best["fit_period_days"],
        "best_bic":best["bic"],"best_fit_matches_truth_for_evaluation":best["candidate_mode_matches_truth"],
        "truth_represented_in_candidates_for_evaluation":any(r["candidate_mode_matches_truth"] for r in fitted),
        "modes_with_delta_bic_le_6":len(close),"period_ambiguous":len(close)>1,
        "best_fit_bound_hits":best["bound_hits"],"seconds":round(time.time()-started,1)})
    pd.DataFrame(candidate_rows).to_csv(OUT/"day9_3_binary_alias_candidate_scores.csv",index=False)
    pd.DataFrame(case_rows).to_csv(OUT/"day9_3_binary_alias_case_summary.csv",index=False)
    print(f"  best={best['fit_period_days']:.6f} d | ΔBIC≤6 modes={len(close)} | evaluation hit={best['candidate_mode_matches_truth']} | {time.time()-started:.1f}s",flush=True)

candidate_df=pd.DataFrame(candidate_rows)
case_df=pd.DataFrame(case_rows)
candidate_df.to_csv(OUT/"day9_3_binary_alias_candidate_scores.csv",index=False)
case_df.to_csv(OUT/"day9_3_binary_alias_case_summary.csv",index=False)

# Plot candidate score against fitted period; truth is shown only for evaluation.
fig,axes=plt.subplots(3,4,figsize=(16,10),constrained_layout=True)
for ax,row in zip(axes.flat,CASE_ROWS):
    s=candidate_df[candidate_df.simulation_row==int(row)].sort_values("delta_bic")
    truth=float(parameters.iloc[int(row)]["period_days"])
    ax.scatter(s.fit_period_days,s.delta_bic,s=20,alpha=.8)
    ax.axvline(truth,color="crimson",ls="--",lw=1,label="known truth (evaluation)")
    ax.axhline(6,color="gray",ls=":",lw=1)
    ax.set(title=f"row {row} | best {s.iloc[0].fit_period_days:.3f} d",
           xlabel="Fitted candidate period (days)",ylabel="ΔBIC")
    ax.legend(fontsize=7)
fig.suptitle("BLS period and half/double aliases")
fig.savefig(OUT/"day9_3_binary_alias_mode_scores.png",dpi=150)
plt.show()

summary={"created_utc":datetime.now(timezone.utc).isoformat(),
 "simulation_root":str(BINARY_DIR.parent),
 "scope":"12 targeted synthetic binary cases; BLS peaks and half/double aliases ranked by deterministic profile likelihood/BIC; no posterior sampling.",
 "completed_cases":int(len(case_df)),
 "truth_in_candidate_set_count_evaluation_only":int(case_df.truth_represented_in_candidates_for_evaluation.sum()),
 "best_bic_matches_truth_count_evaluation_only":int(case_df.best_fit_matches_truth_for_evaluation.sum()),
 "ambiguous_cases_delta_bic_le_6":int(case_df.period_ambiguous.sum()),
 "outputs":["day9_3_binary_alias_candidate_scores.csv","day9_3_binary_alias_case_summary.csv","day9_3_binary_alias_mode_scores.png"],
 "limits":["BIC is not a marginal likelihood or posterior model probability.",
 "Fixed-band BLS peaks plus half/double aliases do not guarantee every possible mode was searched.",
 "Known truth is used only for evaluation, never mode selection.",
 "This targeted synthetic sample does not establish population coverage or real-star validity."]}
(OUT/"day9_3_binary_alias_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(case_df.to_string(index=False))
print(json.dumps(summary,indent=2))
print("Outputs:",OUT)
